In [1]:
# !pip install pyspark
# !pip install -U -q Pydrive
# !apt install openjdk-8-jdk-headless -qq
# import os
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [2]:
!java -version

openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment (build 17.0.19+10-1-22.04.2-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.19+10-1-22.04.2-Ubuntu, mixed mode, sharing)


In [3]:
import pyspark
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType
from pyspark import SparkContext, SparkConf
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator,MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

In [4]:
conf = SparkConf()
conf.setAppName("AdultDatasetRandomForest")
conf.setMaster('local[*]')
conf.set("spark.driver.memory", "2G")
conf.set("spark.driver.maxResultSize", "2g")
conf.set("spark.executor.memory", "1G")
spark = SparkSession.builder.getOrCreate()

## Step 1 Load the dataset
Because the dataset does not include column names, create a schema to assign column names and datatypes.

In [5]:
!wget -O adult.csv https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data

--2026-06-14 22:53:46--  https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘adult.csv’

adult.csv               [  <=>               ]   3.79M  17.7MB/s    in 0.2s    

2026-06-14 22:53:46 (17.7 MB/s) - ‘adult.csv’ saved [3974305]



In [6]:
schema = """`age` DOUBLE,
`workclass` STRING,
`fnlwgt` DOUBLE,
`education` STRING,
`education_num` DOUBLE,
`marital_status` STRING,
`occupation` STRING,
`relationship` STRING,
`race` STRING,
`sex` STRING,
`capital_gain` DOUBLE,
`capital_loss` DOUBLE,
`hours_per_week` DOUBLE,
`native_country` STRING,
`income` STRING"""

# df = spark.read.csv("adult.csv", schema=schema)
# df.show(n=20)
adult_df = spark.read.format('csv').option("header","true").schema(schema).load("adult.csv")
adult_df.show(5)

+----+-----------------+--------+----------+-------------+-------------------+------------------+--------------+------+-------+------------+------------+--------------+--------------+------+
| age|        workclass|  fnlwgt| education|education_num|     marital_status|        occupation|  relationship|  race|    sex|capital_gain|capital_loss|hours_per_week|native_country|income|
+----+-----------------+--------+----------+-------------+-------------------+------------------+--------------+------+-------+------------+------------+--------------+--------------+------+
|50.0| Self-emp-not-inc| 83311.0| Bachelors|         13.0| Married-civ-spouse|   Exec-managerial|       Husband| White|   Male|         0.0|         0.0|          13.0| United-States| <=50K|
|38.0|          Private|215646.0|   HS-grad|          9.0|           Divorced| Handlers-cleaners| Not-in-family| White|   Male|         0.0|         0.0|          40.0| United-States| <=50K|
|53.0|          Private|234721.0|      11th| 

In [7]:
adult_df.select("sex").distinct().show()

+-------+
|    sex|
+-------+
|   Male|
| Female|
+-------+



Randomly split data into training and test sets, and set seed for reproducibility.

It's best to split the data before doing any preprocessing. This allows the test dataset to more closely simulate new data when we evaluate the model.



In [8]:
trainDF, testDF = adult_df.randomSplit([0.8, 0.2], seed=42)
print(trainDF.count())
print(testDF.count())

26076
6484


In [9]:
trainDF\
        .groupBy("education")\
        .count()\
        .sort("count", ascending=False)\
        .show()

+-------------+-----+
|    education|count|
+-------------+-----+
|      HS-grad| 8365|
| Some-college| 5911|
|    Bachelors| 4281|
|      Masters| 1392|
|    Assoc-voc| 1080|
|         11th|  957|
|   Assoc-acdm|  836|
|         10th|  745|
|      7th-8th|  514|
|  Prof-school|  452|
|          9th|  418|
|         12th|  352|
|    Doctorate|  333|
|      5th-6th|  265|
|      1st-4th|  134|
|    Preschool|   41|
+-------------+-----+



we explore three fundamental concepts in MLlib machine learning: **Transformers**, **Estimators**, and **Pipelines**.

A **Transformer** takes a DataFrame as input and produces a new DataFrame as output. It applies predefined transformations without learning parameters. Typically, **Transformers** are used to preprocess data before training a model or making predictions with a trained MLlib model, invoked using the .transform() method.

An **Estimator** learns parameters from a DataFrame using the .fit() method and produces a Model, which itself is a **Transformer**. Unlike **Transformers**, **Estimators** derive parameters based on the input data, enabling them to adapt to specific datasets.

A Pipeline amalgamates multiple stages into a cohesive workflow that can be executed seamlessly. Constructing a machine learning model often involves configuring numerous sequential tasks. Pipelines automate this orchestration, facilitating efficient and reproducible model creation and deployment.

## Step 2: Data Preprocessing

Data preprocessing is essential to prepare the dataset for model training. This involves handling missing values, encoding categorical features, and assembling features into a format suitable for machine learning models.

In [10]:
categoricalColumns = ["workclass", "education", "marital_status", "occupation", "relationship", "race", "sex", "native_country"]

indexers = [StringIndexer(inputCol=col, outputCol=col + "_index", handleInvalid="keep") for col in categoricalColumns]

encoders = [OneHotEncoder(inputCol=col + "_index", outputCol=col + "_encoded") for col in categoricalColumns]

assemblerInputs = [col + "_encoded" for col in categoricalColumns] + ["age", "fnlwgt", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
assembler = VectorAssembler(inputCols=assemblerInputs, outputCol="features")

labelIndexer = StringIndexer(inputCol="income", outputCol="income_index")

# Pipeline
pipeline = Pipeline(stages=indexers + encoders + [assembler, labelIndexer])
pipelineModel = pipeline.fit(adult_df)
preprocessedData = pipelineModel.transform(adult_df).select("features", "income_index")


In [11]:
preprocessedData
distinct_values = preprocessedData.select("income_index").distinct().rdd.map(lambda row: row[0]).collect()
distinct_values

[0.0, 1.0]

**StringIndexer**: Converts categorical columns into numerical indices. This transformation is necessary because most machine learning algorithms  require numerical input.

**VectorAssembler**: Combines indexed categorical columns and numeric columns into a single feature vector. This is the format Spark ML models expect for input features.

**Pipeline**: Combines multiple stages of transformations into a single pipeline for easier management and application to new data.
Selecting Columns: After preprocessing, we select only the features and income_index columns for model training.

## Step 3: Train-Test Split
Splitting the dataset into training and testing sets allows us to evaluate the model's performance on unseen data.

In [12]:
train_data, test_data = preprocessedData.randomSplit([0.8, 0.2], seed=12345)
print(f"Number of train data:{train_data.count()}")
print(f"Number of test data:{test_data.count()}")

Number of train data:25953
Number of test data:6607


In [13]:
train_data

DataFrame[features: vector, income_index: double]

## Step 4: Model Training and Hyperparameter Tuning
In this step, we define a Random Forest classifier, specify a parameter grid for hyperparameter tuning, and use CrossValidator to find the best combination of hyperparameters.

In [14]:
rf = RandomForestClassifier(labelCol="income_index", featuresCol="features")
evaluator = BinaryClassificationEvaluator(labelCol="income_index", rawPredictionCol="rawPrediction", metricName="areaUnderROC")


In [15]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

param_grid = (ParamGridBuilder()
              .addGrid(rf.numTrees, [10, 20, 30])
              .addGrid(rf.maxDepth, [5, 10])
              .addGrid(rf.maxBins,[50,100])
              .build())
crossval = CrossValidator(estimator=rf,
                          estimatorParamMaps=param_grid,
                          evaluator=evaluator,
                          numFolds=3)
cvModel = crossval.fit(train_data)
bestModel = cvModel.bestModel


**RandomForestClassifier**: Initializes a Random Forest classifier with specified labelCol (target variable) and featuresCol (input features).

**ParamGridBuilder**: Defines a grid of parameters (numTrees and maxDepth in this case) to be searched during hyperparameter tuning.

**CrossValidator**: Performs k-fold cross-validation (numFolds=3 here) to find the best model configuration based on the specified evaluator (BinaryClassificationEvaluator in this case).

**Model Evaluation**: Computes the accuracy metric on the test set using the BinaryClassificationEvaluator to assess how well the model generalizes to new data.

In [16]:
predictions = bestModel.transform(train_data)
bestAccuracy = evaluator.evaluate(predictions)
print(f"Best Test Area Under ROC: {bestAccuracy}")
print(f"Best Param (numTrees): {bestModel.getNumTrees}")
print(f"Best Param (maxDepth): {bestModel.getMaxDepth()}")
print(f"Best Param (maxBins): {bestModel.getMaxBins()}")

Best Test Area Under ROC: 0.9215624242244833
Best Param (numTrees): 30
Best Param (maxDepth): 10
Best Param (maxBins): 100


# Case Study: Water Quality  

Access to safe drinking water is essential for health, a basic human right, and a crucial component of effective health protection policies. Ensuring safe drinking water is an important issue for health and development at national, regional, and local levels. Research in various regions has demonstrated that investments in water supply and sanitation can yield significant economic benefits. These benefits arise from reduced adverse health effects and lowered healthcare costs, which outweigh the costs of implementing the necessary interventions.

In this exercise, we will use PySpark in developing a Random Forest model to analyze a dataset related to safe drinking water. The goal is to predict and understand the factors that contribute to safe drinking water access. This analysis will provide insights into the potential health and economic benefits of investing in water supply and sanitation infrastructure.

In [17]:
!wget -O water_potability.csv https://drive.google.com/file/d/1g39BFsY2uWpkEGIkS9A8cnwK4L70vGg7/view?usp=sharing

--2026-06-14 22:57:17--  https://drive.google.com/file/d/1g39BFsY2uWpkEGIkS9A8cnwK4L70vGg7/view?usp=sharing
Resolving drive.google.com (drive.google.com)... 142.250.99.100, 142.250.99.101, 142.250.99.102, ...
Connecting to drive.google.com (drive.google.com)|142.250.99.100|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘water_potability.csv’

water_potability.cs     [ <=>                ]  68.62K  --.-KB/s    in 0.002s  

2026-06-14 22:57:17 (37.1 MB/s) - ‘water_potability.csv’ saved [70269]



In [18]:
### [TODO]

Let's implement the following steps:
1. **Data Loading and Exploration**: Load the dataset into a PySpark DataFrame and explore its structure and contents.
2. **Data Preprocessing**: Clean and preprocess the data to make it suitable for model training. This may include handling missing values, encoding categorical features, and scaling numerical features.
3. **Model Training**: Train the Random Forest model using the prepared dataset.
4. **Model Evaluation**: Evaluate the performance of the model using appropriate metrics and validate its effectiveness in predicting safe drinking water access.
5. **Insights and Conclusions**: Analyze the results and draw conclusions about the factors influencing safe drinking water access and the potential benefits of investments in water supply and sanitation.

Let's get started!